#Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Object Detection

In [ ]:
import cv2
import numpy as np
from keras.models import load_model
from keras.losses import mean_squared_error

model_path = "/content/ev_pareev_mobilenet_mobileprae.keras"
class_label = ["Artifact", "Ev eggs"]

def mse(y_true, y_pred):
  return mean_squared_error(y_true, y_pred)

model = load_model(model_path, custom_objects={'mse': mse})

def drawbox(img, label, a, b, c, d, color):
  text_color = (6, 64, 43)
  image = cv2.rectangle(img, (c, a), (d, b), color, 5)
  image = cv2.putText(image, label, (c, a - 10), cv2.FONT_HERSHEY_TRIPLEX, 3, text_color, 2)
  return image

def compute_iou(box1, box2):
  y1 = max(box1[0], box2[0])
  y2 = min(box1[1], box2[1])
  x1 = max(box1[2], box2[2])
  x2 = min(box1[3], box2[3])
  inter_w = max(0, x2 - x1)
  inter_h = max(0, y2 - y1)
  inter_area = inter_w * inter_h
  box1_area = (box1[1] - box1[0]) * (box1[3] - box1[2])
  box2_area = (box2[1] - box2[0]) * (box2[3] - box2[2])
  union_area = box1_area + box2_area - inter_area
  if union_area == 0:
    return 0
  return inter_area / union_area

def nms(detections, iou_threshold):
  nms_dets = []
  for class_idx in set([d['class_idx'] for d in detections]):
    class_dets = [d for d in detections if d['class_idx'] == class_idx]
    class_dets = sorted(class_dets, key=lambda x: x['score'], reverse=True)
    keep = []
    while class_dets:
      curr = class_dets.pop(0)
      keep.append(curr)
      class_dets = [
        d for d in class_dets
        if compute_iou(curr['bbox'], d['bbox']) < iou_threshold
      ]
    nms_dets.extend(keep)
  return nms_dets

def merge_connected_boxes_by_class(detections, merge_iou_threshold):
  merged = []
  for class_idx in set([d['class_idx'] for d in detections]):
    class_dets = [d for d in detections if d['class_idx'] == class_idx]
    used = set()
    groups = []
    for i, det in enumerate(class_dets):
      if i in used:
        continue
      group = [det]
      used.add(i)
      changed = True
      while changed:
        changed = False
        for j, other in enumerate(class_dets):
          if j in used:
            continue
          if any(compute_iou(d['bbox'], other['bbox']) > merge_iou_threshold for d in group):
            group.append(other)
            used.add(j)
            changed = True
      groups.append(group)
    for group in groups:
      tops = [d['bbox'][0] for d in group]
      bottoms = [d['bbox'][1] for d in group]
      lefts = [d['bbox'][2] for d in group]
      rights = [d['bbox'][3] for d in group]
      merged_box = [min(tops), max(bottoms), min(lefts), max(rights)]
      max_score = max(d['score'] for d in group)
      merged.append({"bbox": merged_box, "class_idx": class_idx, "score": max_score})
  return merged

def ObjectDet(filepath, threshold, nms_threshold, merge_iou_threshold):
  img = cv2.imread(filepath)
  box_size_y, box_size_x, step_size = 500, 500, 50
  resize_input_y, resize_input_x = 64, 64
  img_h, img_w = img.shape[:2]

  coords = []
  patches = []
  for i in range(0, img_h - box_size_y + 1, step_size):
    for j in range(0, img_w - box_size_x + 1, step_size):
      img_patch = img[i:i+box_size_y, j:j+box_size_x]
      brightness = np.mean(cv2.cvtColor(img_patch, cv2.COLOR_BGR2GRAY))
      if brightness < 50:
        continue
      img_patch = cv2.resize(img_patch, (resize_input_y, resize_input_x), interpolation=cv2.INTER_AREA)
      patches.append(img_patch)
      coords.append((i, j))

  patches = np.array(patches)
  y_out = model.predict(patches, batch_size=64, verbose=0)
  detections = []
  for idx, pred in enumerate(y_out):
    for class_idx in range(len(class_label)):
      score = pred[class_idx]
      if score > threshold and class_idx != 0:
        a, c = coords[idx]
        b, d = a + box_size_y, c + box_size_x
        detections.append({"bbox": [a, b, c, d], "score": float(score), "class_idx": class_idx})

  nms_detections = nms(detections, iou_threshold=nms_threshold)
  if merge_iou_threshold is not None and merge_iou_threshold > 0:
    merged_detections = merge_connected_boxes_by_class(nms_detections, merge_iou_threshold=merge_iou_threshold)
  else:
    merged_detections = nms_detections

  img_output = img.copy()
  colors = [(0,255,0), (255,0,0), (0,0,255), (0,255,255), (255,0,255), (255,255,0)]
  for det in merged_detections:
    a, b, c, d = det['bbox']
    class_idx = det['class_idx']
    label = f"{class_label[class_idx]}: {det['score']:.2f}"
    color = colors[class_idx % len(colors)]
    img_output = drawbox(img_output, label, a, b, c, d, color)
  return img_output


In [ ]:
import os
import datetime
import cv2
import matplotlib.pyplot as plt

input_folder = '/content/drive/MyDrive/ev_pare/02_object_detection/'
output_folder = '/content/drive/MyDrive/ev_pare/04_boundingbox_model_IoU/'


for filename in os.listdir(input_folder):
  image_path = os.path.join(input_folder, filename)
  print(filename)
  print("Start time:", datetime.datetime.now().time())
  img_output = ObjectDet(image_path, 0.95, 0.3, 0.5)
 # cv2.imwrite(output_folder + f'{location}' + filename, img_output) #เซฟไฟล์
  cv2.imwrite(output_folder + filename, img_output)
  print("End time:", datetime.datetime.now().time())

print("Finish time:", datetime.datetime.now().time())

IMG_20250610_140113.jpg
Start time: 03:32:38.051903
End time: 03:33:46.354755
IMG_20250610_140529.jpg
Start time: 03:33:46.354940
End time: 03:35:14.680103
IMG_20250610_141500.jpg
Start time: 03:35:14.680345
End time: 03:36:43.283095
IMG_20250610_141811.jpg
Start time: 03:36:43.283292
End time: 03:38:11.222816
IMG_20250610_142612.jpg
Start time: 03:38:11.223132
End time: 03:39:18.986595
IMG_20250610_142903.jpg
Start time: 03:39:18.986773
End time: 03:40:25.885296
IMG_20250610_142954.jpg
Start time: 03:40:25.885467
End time: 03:41:53.004565
IMG_20250610_143236.jpg
Start time: 03:41:53.004771
End time: 03:43:04.162948
IMG_20250610_143321.jpg
Start time: 03:43:04.163173
End time: 03:44:31.642396
IMG_20250610_143349.jpg
Start time: 03:44:31.642574
End time: 03:45:36.463420
IMG_20250610_143448.jpg
Start time: 03:45:36.463584
End time: 03:46:39.134937
IMG_20250610_143617.jpg
Start time: 03:46:39.135112
End time: 03:47:45.284766
IMG_20250610_143702.jpg
Start time: 03:47:45.284948
End time: 03

# แปลงภาพขาวดำ

## expert to white-black

In [ ]:
import cv2
import numpy as np
import os

# 1. กำหนดโฟลเดอร์ (ปรับตามต้องการ)
input_folder = '/content/drive/MyDrive/ev_pare/03_boundingbox_export'
output_folder = '/content/drive/MyDrive/ev_pare/05_boundingbox_expert'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 2. ตั้งค่าช่วงสีของกรอบ (สีแดง)
lower_bound = np.array([150, 0, 0])
upper_bound = np.array([255, 40, 40])

for filename in os.listdir(input_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(input_folder, filename)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.inRange(img_rgb, lower_bound, upper_bound)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        binary_output = np.zeros(mask.shape, dtype=np.uint8)

        if len(contours) > 0:
            # --- ตรรกะการรวบกรอบเป็นใบเดียว ---
            all_x = []
            all_y = []
            all_w = []
            all_h = []

            for c in contours:
                if cv2.contourArea(c) > 5:
                    x, y, w, h = cv2.boundingRect(c)
                    all_x.append(x)
                    all_y.append(y)
                    all_x.append(x + w) # เก็บจุดสิ้นสุดด้านขวา
                    all_y.append(y + h) # เก็บจุดสิ้นสุดด้านล่าง

            if all_x and all_y:
                # หาขอบเขตที่กว้างที่สุด (รวบทุกกรอบ)
                x_min = min(all_x)
                y_min = min(all_y)
                x_max = max(all_x)
                y_max = max(all_y)

                # วาดกล่องใบเดียวที่คลุมพื้นที่ทั้งหมด (ถมขาวเต็ม)
                cv2.rectangle(binary_output, (x_min, y_min), (x_max, y_max), 255, thickness=-1)

        # 5. บันทึกผลลัพธ์โดยใช้ชื่อเดิม
        save_path = os.path.join(output_folder, filename)
        cv2.imwrite(save_path, binary_output)
        print(f"รวบกรอบเสร็จสิ้น: {filename}")

print("\n--- เรียบร้อย! ทุกรูปที่มีหลายกรอบถูกรวบเป็นกรอบใหญ่ใบเดียวแล้วครับ ---")

รวบกรอบเสร็จสิ้น: IMG_20250610_140113.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_140529.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_141500.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_141811.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_142612.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_142903.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_142954.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143236.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143321.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143349.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143448.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143617.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143702.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144025.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144053.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144125.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144411.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144512.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144625.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144822.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144838.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144856.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144920.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_145

## model to white-black

In [ ]:
import cv2
import numpy as np
import os

# 1. กำหนดโฟลเดอร์ (ปรับตามต้องการ)
input_folder = '/content/drive/MyDrive/ev_pare/04_boundingbox_model'
output_folder = '/content/drive/MyDrive/ev_pare/06_boundingbox_model'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 2. ตั้งค่าช่วงสีของกรอบ (สีแดง)

lower_bound = np.array([0, 0, 200])
upper_bound = np.array([100, 50, 255])

for filename in os.listdir(input_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(input_folder, filename)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.inRange(img_rgb, lower_bound, upper_bound)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        binary_output = np.zeros(mask.shape, dtype=np.uint8)

        if len(contours) > 0:
            # --- ตรรกะการรวบกรอบเป็นใบเดียว ---
            all_x = []
            all_y = []
            all_w = []
            all_h = []

            for c in contours:
                if cv2.contourArea(c) > 5:
                    x, y, w, h = cv2.boundingRect(c)
                    all_x.append(x)
                    all_y.append(y)
                    all_x.append(x + w) # เก็บจุดสิ้นสุดด้านขวา
                    all_y.append(y + h) # เก็บจุดสิ้นสุดด้านล่าง

            if all_x and all_y:
                # หาขอบเขตที่กว้างที่สุด (รวบทุกกรอบ)
                x_min = min(all_x)
                y_min = min(all_y)
                x_max = max(all_x)
                y_max = max(all_y)

                # วาดกล่องใบเดียวที่คลุมพื้นที่ทั้งหมด (ถมขาวเต็ม)
                cv2.rectangle(binary_output, (x_min, y_min), (x_max, y_max), 255, thickness=-1)

        # 5. บันทึกผลลัพธ์โดยใช้ชื่อเดิม
        save_path = os.path.join(output_folder, filename)
        cv2.imwrite(save_path, binary_output)
        print(f"รวบกรอบเสร็จสิ้น: {filename}")

print("\n--- เรียบร้อย! ทุกรูปที่มีหลายกรอบถูกรวบเป็นกรอบใหญ่ใบเดียวแล้วครับ ---")

รวบกรอบเสร็จสิ้น: IMG_20250610_140113.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_140529.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_141500.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_141811.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_142612.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_142903.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_142954.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143236.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143321.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143349.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143448.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143617.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_143702.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144025.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144053.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144125.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144411.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144512.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144625.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144822.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144838.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144856.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_144920.jpg
รวบกรอบเสร็จสิ้น: IMG_20250610_145

## IoU

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd  # เพิ่ม pandas สำหรับจัดการ Excel

def findbox(img, lower_rgb, upper_rgb):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = cv2.inRange(img_rgb, lower_rgb, upper_rgb)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    # รวมทุก Contour เป็นกล่องเดียว (Single Box Logic)
    all_points = np.concatenate(contours)
    x, y, w, h = cv2.boundingRect(all_points)
    return [x, y, x + w, y + h]

def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
    boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)

    if interArea == 0:
        return 0.0

    return interArea / float(boxAArea + boxBArea - interArea)

# --- ตั้งค่า Path ---
exp_folder = '/content/drive/MyDrive/ev_pare/05_boundingbox_expert'
mod_folder = '/content/drive/MyDrive/ev_pare/06_boundingbox_model'
excel_output_path = '/content/drive/MyDrive/ev_pare/iou_results.xlsx'

# สีกรอบ (ในที่นี้คือสีขาวตามไฟล์ Binary ที่เราเพิ่งสร้างไป)
white_lower = np.array([255, 255, 255])
white_upper = np.array([255, 255, 255])

# สร้าง List เพื่อเก็บข้อมูลสำหรับ Excel
results_list = []

for fname in os.listdir(exp_folder):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    exp_path = os.path.join(exp_folder, fname)
    matched_files = [f for f in os.listdir(mod_folder) if f.endswith(fname)]

    if not matched_files:
        print(f"ไม่พบภาพ model ที่ตรงกับ: {fname}")
        continue

    mod_path = os.path.join(mod_folder, matched_files[0])

    img_exp = cv2.imread(exp_path)
    img_mod = cv2.imread(mod_path)

    bbox_exp = findbox(img_exp, white_lower, white_upper)
    bbox_mod = findbox(img_mod, white_lower, white_upper)

    iou = 0.0
    if bbox_exp is not None and bbox_mod is not None:
        iou = calculate_iou(bbox_exp, bbox_mod)

    # เก็บข้อมูลลงใน Dictionary
    results_list.append({
        'File Name': fname,
        'Expert BBox': str(bbox_exp),
        'Model BBox': str(bbox_mod),
        'IoU Score': round(iou, 4)
    })

    print(f"Processed: {fname} | IoU: {iou:.4f}")

# --- บันทึกเป็นไฟล์ Excel ---
df = pd.DataFrame(results_list)
df.to_excel(excel_output_path, index=False)

print(f"\n--- เรียบร้อย! เซฟไฟล์ Excel แล้วที่: {excel_output_path} ---")

Processed: IMG_20250610_161839.jpg | IoU: 0.5236
Processed: IMG_20250610_161939.jpg | IoU: 0.5956
Processed: IMG_20250610_162142.jpg | IoU: 0.5298
Processed: IMG_20250610_162957.jpg | IoU: 0.5353
Processed: IMG_20250610_163333.jpg | IoU: 0.5336
Processed: IMG_20250610_163620.jpg | IoU: 0.5929
Processed: IMG_20250610_163655.jpg | IoU: 0.6527
Processed: IMG_20250610_164342.jpg | IoU: 0.2047
Processed: IMG_20250611_135933.jpg | IoU: 0.6167
Processed: IMG_20250611_140149.jpg | IoU: 0.5862
Processed: IMG_20250611_140445.jpg | IoU: 0.6024
Processed: IMG_20250611_141229.jpg | IoU: 0.7836
Processed: IMG_20250611_141338.jpg | IoU: 0.8181
Processed: IMG_20250611_141748.jpg | IoU: 0.6768
Processed: IMG_20250611_141850.jpg | IoU: 0.6400
Processed: IMG_20250611_142126.jpg | IoU: 0.5939
Processed: IMG_20250611_142208.jpg | IoU: 0.0900
Processed: IMG_20250611_142245.jpg | IoU: 0.6919
Processed: IMG_20250611_142329.jpg | IoU: 0.5661
Processed: IMG_20250611_142343.jpg | IoU: 0.6226
Processed: IMG_20250